# EEI6366 Lab 01: Big Data Processing with PySpark

**Dataset:** Open University Student Performance (`open_university_student_performance.csv`)  
**Size:** 150,500 rows × 9 columns  
**Source:** Generated / provided for EEI6366 (Open University of Sri Lanka student performance across centers and courses)

This notebook fulfills **Step 2 – Prepare and Do the Lab**:
1. Study Getting Started with PySpark (completed)
2. Use a dataset with ≥ 100,000 rows
3. Explore with **PySpark DataFrame API** (≥ 3 operations)
4. Explore with **Spark SQL** (≥ 3 operations)

Columns: `student_id`, `course_code`, `academic_year`, `center`, `continuous_assessment_mark`, `final_exam_mark`, `overall_mark`, `grade`, `status`

## 1. Install & Initialize PySpark (Google Colab style)

In [ ]:
# Run this cell only if you are on Google Colab and PySpark is not already available
!pip install -q pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (SparkSession
         .builder
         .appName("EEI6366-Lab01-Student-Performance")
         .master("local[*]")
         .getOrCreate())

spark

## 2. Load the Dataset

In [ ]:
# Upload the CSV to Colab or place it in the working directory
# In Colab: from google.colab import files; files.upload()

df = spark.read.csv(
    "open_university_student_performance.csv",
    header=True,
    inferSchema=True
)

print("Total rows:", df.count())
print("Total columns:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)

## 3. PySpark DataFrame API Operations (≥ 3)

### Operation 1 – Basic exploration & descriptive statistics

In [ ]:
# Schema already printed above
# Summary statistics for numeric columns
df.select(
    "continuous_assessment_mark",
    "final_exam_mark",
    "overall_mark"
).describe().show()

# Distinct values for categorical columns
print("Unique courses:", df.select("course_code").distinct().count())
print("Unique centers:", df.select("center").distinct().count())
print("Academic years:")
df.select("academic_year").distinct().orderBy("academic_year").show()

### Operation 2 – Filtering + Ordering

In [ ]:
# Students who failed the EEI6366 - Big Data Processing course, ordered by overall_mark ascending
failed_bigdata = (df
    .filter(
        (F.col("course_code") == "EEI6366 - Big Data Processing") &
        (F.col("status") == "Fail")
    )
    .orderBy(F.col("overall_mark").asc())
    .select("student_id", "center", "academic_year",
            "continuous_assessment_mark", "final_exam_mark",
            "overall_mark", "grade")
)

print("Number of Failures in EEI6366:", failed_bigdata.count())
failed_bigdata.show(10, truncate=False)

# High performers (overall_mark >= 80) from Colombo center
high_colombo = df.filter(
    (F.col("center") == "Colombo") &
    (F.col("overall_mark") >= 80)
).orderBy(F.desc("overall_mark"))

print("\nHigh performers in Colombo (>=80):", high_colombo.count())
high_colombo.show(5)

### Operation 3 – Aggregations (GroupBy + Agg)

In [ ]:
# Average marks and pass rate by Center
center_stats = (df
    .groupBy("center")
    .agg(
        F.count("*").alias("total_students"),
        F.round(F.avg("overall_mark"), 2).alias("avg_overall_mark"),
        F.round(F.avg("continuous_assessment_mark"), 2).alias("avg_ca"),
        F.round(F.avg("final_exam_mark"), 2).alias("avg_exam"),
        F.round(
            F.sum(F.when(F.col("status") == "Pass", 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("pass_rate_pct")
    )
    .orderBy(F.desc("avg_overall_mark"))
)

print("Performance by Regional Center:")
center_stats.show(truncate=False)

# Average marks and pass rate by Course
course_stats = (df
    .groupBy("course_code")
    .agg(
        F.count("*").alias("enrollments"),
        F.round(F.avg("overall_mark"), 2).alias("avg_overall"),
        F.round(
            F.sum(F.when(F.col("status") == "Pass", 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("pass_rate_pct")
    )
    .orderBy(F.desc("avg_overall"))
)

print("\nPerformance by Course:")
course_stats.show(truncate=False)

### Bonus Operation – Derived column + Window function (ranking)

In [ ]:
# Create a new column: performance gap (CA - Exam)
df_with_gap = df.withColumn(
    "ca_exam_gap",
    F.round(F.col("continuous_assessment_mark") - F.col("final_exam_mark"), 1)
)

# Rank students within each course by overall_mark (top performers)
window_spec = Window.partitionBy("course_code").orderBy(F.desc("overall_mark"))

ranked = (df_with_gap
    .withColumn("rank_in_course", F.rank().over(window_spec))
    .filter(F.col("rank_in_course") <= 3)   # Top 3 per course
    .select("course_code", "student_id", "center", "overall_mark",
            "grade", "ca_exam_gap", "rank_in_course")
    .orderBy("course_code", "rank_in_course")
)

print("Top 3 students per course:")
ranked.show(20, truncate=False)

## 4. Spark SQL Operations (≥ 3)

First register the DataFrame as a temporary view.

In [ ]:
df.createOrReplaceTempView("student_performance")

# Verify
spark.sql("SELECT COUNT(*) AS total_rows FROM student_performance").show()

### SQL Operation 1 – Filtering + Aggregation

In [ ]:
# Pass rate and average overall mark by Academic Year and Center
spark.sql("""
SELECT
    academic_year,
    center,
    COUNT(*) AS students,
    ROUND(AVG(overall_mark), 2) AS avg_overall,
    ROUND(SUM(CASE WHEN status = 'Pass' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pass_rate_pct
FROM student_performance
GROUP BY academic_year, center
ORDER BY academic_year, avg_overall DESC
""").show(30, truncate=False)

### SQL Operation 2 – Conditional logic (CASE) + Grade distribution

In [ ]:
# Grade distribution for the course EEI6366 - Big Data Processing
spark.sql("""
SELECT
    grade,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM student_performance
WHERE course_code = 'EEI6366 - Big Data Processing'
GROUP BY grade
ORDER BY
    CASE grade
        WHEN 'A+' THEN 1 WHEN 'A' THEN 2 WHEN 'A-' THEN 3
        WHEN 'B+' THEN 4 WHEN 'B' THEN 5 WHEN 'B-' THEN 6
        WHEN 'C+' THEN 7 WHEN 'C' THEN 8 WHEN 'C-' THEN 9
        WHEN 'D+' THEN 10 WHEN 'D' THEN 11 WHEN 'F' THEN 12
        ELSE 13
    END
""").show()

### SQL Operation 3 – Window function (moving / ranking style)

In [ ]:
# For each student in a center, show their overall_mark and the average overall_mark of that center
# plus the difference from the center average (using window)
spark.sql("""
SELECT
    student_id,
    center,
    course_code,
    overall_mark,
    ROUND(AVG(overall_mark) OVER (PARTITION BY center), 2) AS center_avg,
    ROUND(overall_mark - AVG(overall_mark) OVER (PARTITION BY center), 2) AS diff_from_center_avg
FROM student_performance
WHERE course_code = 'EEI6366 - Big Data Processing'
ORDER BY center, overall_mark DESC
LIMIT 20
""").show(truncate=False)

### SQL Operation 4 (extra) – Finding interesting patterns

In [ ]:
# Students whose Continuous Assessment is strong (>=70) but Final Exam is weak (<40)
# – possible exam anxiety or preparation issues
spark.sql("""
SELECT
    student_id,
    course_code,
    center,
    continuous_assessment_mark AS ca,
    final_exam_mark AS exam,
    overall_mark,
    grade,
    status
FROM student_performance
WHERE continuous_assessment_mark >= 70
  AND final_exam_mark < 40
ORDER BY continuous_assessment_mark DESC
LIMIT 15
""").show(truncate=False)

## 5. Key Findings (for Padlet / Presentation)

- Dataset contains **150,500** student-course records across **6 courses**, **6 regional centers** and **3 academic years**.
- Overall pass rate is high (~90%).
- Performance varies by center; some centers show higher average overall marks and pass rates.
- There is a noticeable group of students with strong Continuous Assessment but weak Final Exam performance – useful insight for teaching support.
- Window functions and group-by aggregations make it easy to rank students and compute center/course level KPIs at scale.

## 6. Challenges

- Setting up the Spark environment correctly (especially Java & SPARK_HOME on local machines).
- Inferring schema correctly and handling any missing values (this dataset is clean).
- Choosing meaningful aggregations that produce actionable insights rather than just running random queries.

---
**Repository:** https://github.com/missarii/Big-Data-Processing-with-PySpark  
End of Lab 01 exploration.

In [ ]:
# Stop the Spark session when finished
spark.stop()
print("Spark session stopped.")